In [ ]:
from google.colab import drive
drive.mount('/content/drive')

TRAIN_PATH = "/content/drive/MyDrive/ILLM_LAB_4/train.txt"
VALID_PATH = "/content/drive/MyDrive/ILLM_LAB_4/dev.txt"
TEST_PATH  = "/content/drive/MyDrive/ILLM_LAB_4/test.txt"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# ILLM Lab 5 — Word2Vec, FastText, Doc2Vec, Clustering & Sentence Similarity

In [ ]:
# # If needed, uncomment to install dependencies
!pip install gensim scikit-learn numpy tqdm


In [ ]:
import os
import re
import random
import numpy as np
from tqdm import tqdm

from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
np.random.seed(42)


## 1. Load & Tokenize the Data

A very simple whitespace + punctuation-stripping tokenizer is used below.
If your language needs a dedicated tokenizer (e.g. Indic NLP Library, IndicBERT tokenizer, jieba for Chinese, etc.), swap out `tokenize()` accordingly — the rest of the notebook only depends on this function returning a list of tokens.


In [ ]:
def tokenize(sentence: str):
    """Basic tokenizer — replace with a language-specific tokenizer if needed."""
    sentence = sentence.strip()
    # keep word characters (incl. unicode letters) and split on everything else
    tokens = re.findall(r"\w+", sentence, flags=re.UNICODE)
    return [t.lower() for t in tokens]


def load_sentences(path):
    with open(path, encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
    return lines


train_sentences_raw = load_sentences(TRAIN_PATH)
valid_sentences_raw = load_sentences(VALID_PATH)
test_sentences_raw  = load_sentences(TEST_PATH)

print(f"Train sentences: {len(train_sentences_raw)}")
print(f"Valid sentences: {len(valid_sentences_raw)}")
print(f"Test sentences:  {len(test_sentences_raw)}")

train_tokens = [tokenize(s) for s in tqdm(train_sentences_raw, desc="Tokenizing train")]
valid_tokens = [tokenize(s) for s in tqdm(valid_sentences_raw, desc="Tokenizing valid")]
test_tokens  = [tokenize(s) for s in tqdm(test_sentences_raw,  desc="Tokenizing test")]

# Drop any empty token lists (keep raw sentence lists in sync)
def drop_empty(tok_list, raw_list):
    pairs = [(t, r) for t, r in zip(tok_list, raw_list) if len(t) > 0]
    toks, raws = zip(*pairs) if pairs else ([], [])
    return list(toks), list(raws)

train_tokens, train_sentences_raw = drop_empty(train_tokens, train_sentences_raw)
valid_tokens, valid_sentences_raw = drop_empty(valid_tokens, valid_sentences_raw)
test_tokens,  test_sentences_raw  = drop_empty(test_tokens,  test_sentences_raw)


Train sentences: 1000000
Valid sentences: 1000
Test sentences:  1000


Tokenizing test: 100%|██████████| 1000/1000 [00:00<00:00, 12268.42it/s]


## 2a. Train Word2Vec

Following the gensim tutorial: https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html


In [ ]:
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 5
WORKERS = os.cpu_count() or 4
EPOCHS = 3

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=WORKERS,
    sg=1,          # 1 = skip-gram, 0 = CBOW
    epochs=EPOCHS,
)

w2v_model.save("/content/drive/MyDrive/ILLM_LAB_5/word2vec.model")
print("Word2Vec vocabulary size:", len(w2v_model.wv))


Word2Vec vocabulary size: 14912


In [ ]:
# Sanity check — most similar words to a sample word from the vocabulary
sample_word = w2v_model.wv.index_to_key[0]
print(f"Most similar to '{sample_word}':")
for word, score in w2v_model.wv.most_similar(sample_word, topn=5):
    print(f"  {word}: {score:.4f}")


Most similar to 'ર':
  ન: 0.7140
  પરચમ: 0.7019
  હલચલન: 0.6998
  ૪૮૦૦: 0.6974
  અછડત: 0.6968


## 2b. Train FastText

Following the gensim tutorial: https://radimrehurek.com/gensim/auto_examples/tutorials/run_fasttext.html


In [ ]:
fasttext_model = FastText(
    sentences=train_tokens,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=WORKERS,
    sg=1,
    epochs=EPOCHS,
)

fasttext_model.save("/content/drive/MyDrive/ILLM_LAB_5/fasttext.model")
print("FastText vocabulary size:", len(fasttext_model.wv))


FastText vocabulary size: 14912


In [ ]:
# Sanity check — FastText can even handle out-of-vocabulary/misspelled words
print(f"Most similar to '{sample_word}' (FastText):")
for word, score in fasttext_model.wv.most_similar(sample_word, topn=5):
    print(f"  {word}: {score:.4f}")


Most similar to 'ર' (FastText):
  તઘલખ: 0.7259
  એ1: 0.7196
  રઓ: 0.7186
  ટù: 0.7177
  તઘ: 0.7144


## 2c. Train Doc2Vec

Following the gensim tutorial: https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.html


In [ ]:
tagged_train_docs = [
    TaggedDocument(words=tokens, tags=[str(i)])
    for i, tokens in enumerate(train_tokens)
]

doc2vec_model = Doc2Vec(
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=WORKERS,
    epochs=EPOCHS,
    dm=1,   # 1 = PV-DM, 0 = PV-DBOW
)

doc2vec_model.build_vocab(tagged_train_docs)
doc2vec_model.train(
    tagged_train_docs,
    total_examples=doc2vec_model.corpus_count,
    epochs=doc2vec_model.epochs,
)

doc2vec_model.save("/content/drive/MyDrive/ILLM_LAB_5/doc2vec.model")
print("Doc2Vec trained on", len(tagged_train_docs), "documents")


Doc2Vec trained on 999999 documents


In [ ]:
inferred = doc2vec_model.infer_vector(tagged_train_docs[0].words)
sims = doc2vec_model.dv.most_similar([inferred], topn=3)
print("Self-similarity check (top match should be tag '0'):")
print(sims)


Self-similarity check (top match should be tag '0'):
[('858845', 0.813488245010376), ('995911', 0.8044059872627258), ('890592', 0.798843502998352)]


## 3a. K-Means Clustering of Words

We cluster the Word2Vec word vectors with K-Means, then represent each cluster
by the **20 words whose embeddings are closest to the cluster's centroid**.


In [ ]:
NUM_CLUSTERS = 20  
TOP_WORDS_PER_CLUSTER = 20

vocab_words = w2v_model.wv.index_to_key
word_vectors = w2v_model.wv[vocab_words]  

kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(word_vectors)

print(f"Clustered {len(vocab_words)} words into {NUM_CLUSTERS} clusters.")


Clustered 14912 words into 20 clusters.


In [ ]:
def top_words_for_clusters(vocab_words, word_vectors, kmeans, top_n=20):
    """For each cluster centroid, find the top_n closest words by cosine similarity."""
    centroids = kmeans.cluster_centers_
    sims = cosine_similarity(word_vectors, centroids)  
    cluster_to_words = {}
    for c in range(centroids.shape[0]):
        
        top_idx = np.argsort(-sims[:, c])[:top_n]
        cluster_to_words[c] = [(vocab_words[i], float(sims[i, c])) for i in top_idx]
    return cluster_to_words


cluster_to_words = top_words_for_clusters(
    vocab_words, word_vectors, kmeans, top_n=TOP_WORDS_PER_CLUSTER
)

for c, words in cluster_to_words.items():
    word_str = ", ".join(f"{w}({s:.2f})" for w, s in words)
    print(f"Cluster {c}: {word_str}\n")


Cluster 0: ૪૩(0.93), ૮૬(0.93), ૫૫(0.93), ૫૨(0.93), ૪૬(0.92), ૫૪(0.92), ૫૬(0.92), ૫૯(0.92), ૪૭(0.92), ૮૭(0.92), ૬૮(0.92), ૫૮(0.92), ૬૧(0.92), ૭૯(0.92), ૩૨(0.92), ૬૪(0.92), ૮૮(0.91), ૭૧(0.91), ૮૪(0.91), ૪૨(0.91)

Cluster 1: ઢસડ(0.87), સમસમ(0.87), ઘસડ(0.87), આદતથ(0.86), ખખડધજ(0.85), અણછ(0.85), છપર(0.85), ભરખમ(0.85), ઢક(0.85), ચકરડ(0.84), વમળમ(0.84), આપધ(0.84), ચડભડ(0.84), ઢમ(0.84), હવસખ(0.84), તણખલ(0.84), અપહરણક(0.84), ખડકલ(0.84), મગરન(0.84), સમથક(0.84)

Cluster 2: બગસર(0.89), લખતર(0.87), દમણન(0.86), ઉનહ(0.86), રનગરથ(0.86), કરમસદ(0.86), સનગર(0.86), દરણ(0.85), ઉમલ(0.85), ભરતનગર(0.85), ઈસનપ(0.85), વડનગર(0.85), વગઢબ(0.85), કઠલ(0.85), હળવદમ(0.84), મરખ(0.84), વલભ(0.84), ઇસનપ(0.84), કપડવ(0.84), ઉધમપ(0.83)

Cluster 3: ઉથહ(0.98), ઘરદ(0.98), ર૪મ(0.98), ડટન(0.98), dog(0.98), 847(0.98), âન(0.98), drdoએ(0.98), ફએ(0.98), 1953મ(0.98), 1937મ(0.98), 1610(0.98), સયટ(0.98), right(0.98), cnn(0.98), રહમત(0.98), 1936મ(0.98), 1770(0.98), સહઆર(0.98), sho(0.98)

Cluster 4: ઈપણન(0.90), હલનચલનન(0.88), બળથ(0.88), વ

## 3b. Sentence Representations & Nearest-Sentence Search

For every sentence in the **validation** and **test** sets we build two kinds of vectors:

1. **Averaged word-embedding vector** — the mean of the Word2Vec vectors of its tokens.
2. **Doc2Vec inferred vector** — using `doc2vec_model.infer_vector`.

Then, for each validation/test sentence, we find the **closest training sentence**
(by cosine similarity) under each representation.


In [ ]:
def sentence_vector_avg(tokens, kv):
    """Average word-embedding vector for a tokenized sentence, using a KeyedVectors object."""
    vecs = [kv[w] for w in tokens if w in kv]
    if not vecs:
        return np.zeros(kv.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)


# Pre-compute averaged Word2Vec vectors for all training sentences
train_avg_vectors = np.array(
    [sentence_vector_avg(tokens, w2v_model.wv) for tokens in tqdm(train_tokens, desc="Train avg-vecs")]
)

# Pre-compute Doc2Vec vectors for all training sentences (these are already
# available as doc2vec_model.dv, but re-inferring keeps things consistent)
train_doc2vec_vectors = np.array(
    [doc2vec_model.dv[str(i)] for i in range(len(train_tokens))]
)

print("train_avg_vectors shape:    ", train_avg_vectors.shape)
print("train_doc2vec_vectors shape:", train_doc2vec_vectors.shape)


Train avg-vecs: 100%|██████████| 999999/999999 [01:32<00:00, 10767.77it/s]


train_avg_vectors shape:     (999999, 100)
train_doc2vec_vectors shape: (999999, 100)


In [ ]:
def find_closest_training_sentence(query_vector, train_vectors, train_sentences):
    sims = cosine_similarity(query_vector.reshape(1, -1), train_vectors)[0]
    best_idx = int(np.argmax(sims))
    return best_idx, train_sentences[best_idx], float(sims[best_idx])


def process_split(split_name, split_tokens, split_sentences_raw,
                   train_avg_vectors, train_doc2vec_vectors,
                   train_sentences_raw, w2v_model, doc2vec_model,
                   max_print=10):
    results = []
    for tokens, raw in tqdm(list(zip(split_tokens, split_sentences_raw)), desc=f"Processing {split_name}"):
        # 1) averaged word-embedding representation
        query_avg_vec = sentence_vector_avg(tokens, w2v_model.wv)
        avg_idx, avg_match, avg_score = find_closest_training_sentence(
            query_avg_vec, train_avg_vectors, train_sentences_raw
        )

        # 2) Doc2Vec representation
        query_doc_vec = doc2vec_model.infer_vector(tokens)
        doc_idx, doc_match, doc_score = find_closest_training_sentence(
            query_doc_vec, train_doc2vec_vectors, train_sentences_raw
        )

        results.append({
            "sentence": raw,
            "closest_by_word_embedding": avg_match,
            "word_embedding_score": avg_score,
            "closest_by_doc2vec": doc_match,
            "doc2vec_score": doc_score,
        })
    return results


valid_results = process_split(
    "validation", valid_tokens, valid_sentences_raw,
    train_avg_vectors, train_doc2vec_vectors, train_sentences_raw,
    w2v_model, doc2vec_model,
)

test_results = process_split(
    "test", test_tokens, test_sentences_raw,
    train_avg_vectors, train_doc2vec_vectors, train_sentences_raw,
    w2v_model, doc2vec_model,
)


Processing test: 100%|██████████| 1000/1000 [15:38<00:00,  1.07it/s]


In [ ]:
# Preview a few results from the validation set
for r in valid_results[:5]:
    print("Sentence:               ", r["sentence"])
    print("Closest (word-emb avg): ", r["closest_by_word_embedding"], f'(score={r["word_embedding_score"]:.3f})')
    print("Closest (Doc2Vec):      ", r["closest_by_doc2vec"], f'(score={r["doc2vec_score"]:.3f})')
    print("-" * 80)


Sentence:                શહેરની નાલંદા સોસાયટીમાં રહેતા અને ઇનોવેટીવ સ્કુલમાં ધોરણ ૧૨માં અભ્યાસ કરતો સંદીપ ભરતભાઈ ડવ ઉ.૧૮ બપોરે સાડા બારે રીક્ષામાં બેસી સ્કુલે ગયો હતો આત્મીય કોલેજ પાસે ઉતરી સ્કુલે જતો હતો ત્યારે ન્યુ પરિમલ સ્કુલની શેરીમાં આત્મીય કોલેજમાં ભણતો ઉદયસીહ રાણા મારી સાથે ભણતા મારા મિત્ર મેહુલ હુંબલ અને આદિત્ય સાથે ઝઘડો કર્યો હતો હું દોડીને તેને છોડાવવા જતા ઉદયસિહ અને અન્ય ત્રણ શખસોએ મને ગાળો ભાંડી ઉદયસિહે મને હાથ અને માથામાં છરી મારી દીધી હતી અને ધોકાથી હુમલો કરી ઢીકાપાટુનો માર માર્યો હતો મને લોહી નીકળવા લાગતા અને ટોળું એકઠું થઇ જતા ચારેય ભાગી ગયા હતા મામા અને પિતાને ફેન કરતા મને ૧૦૮ મારફ્તે સિવિલ હોસ્પીટલે ખસેડાયો હતો બનાવને પગલે યુનીવર્સીટી પોલીસ મથકના પીએસઆઈ એ બી વોરા અને સ્ટાફ્ે તપાસ કરી ઉદયસિહ રાણા, ઋતુરાજસિહ જાડેજા, વીરપાલસિહ પરમાર અને સીદાન્સું ચૌહાણ સામે ફ્રિયાદ નોંધી ચારેયને સકંજામાં લઇ આગવીઢબે પુછતાછ શરુ કરી હતી જાહેરમાં છરી-ધોકા ઉડતા અન્ય વિદ્યાર્થીઓમાં પણ નાશભાગ મચી જવા પામી હતી.
Closest (word-emb avg):  પોલીસ અધિકારી અનુસાર ગામમાં આતંકી હોવાની જાણ થતા જ ભારતીય સ

In [ ]:
# Save results to disk (CSV) for the report
import csv

def save_results_csv(results, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        writer.writeheader()
        writer.writerows(results)

save_results_csv(valid_results, "/content/drive/MyDrive/ILLM_LAB_5/validation_nearest_sentences.csv")
save_results_csv(test_results, "/content/drive/MyDrive/ILLM_LAB_5.test_nearest_sentences.csv")
print("Saved ILLM_LAB_5/validation_nearest_sentences.csv and ILLM_LAB_5/test_nearest_sentences.csv")


Saved ILLM_LAB_5/validation_nearest_sentences.csv and ILLM_LAB_5/test_nearest_sentences.csv
